# Hawkes $k$-Point Function — Full Pipeline Demo

**Model:** Nonlinear Hawkes process, 2 populations, delta-function synaptic filter

Set `k` and `max_ell` in the configuration cell below to control which
cumulant and loop orders are computed.  All intermediate results are
cached so subsequent runs with the same parameters are instantaneous.

In [ ]:
import os, sys
# --- depth-robust repo root: walk up until we find the 'pipeline' package ---
_root = os.path.abspath('')
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, 'pipeline')):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)
os.chdir(os.path.join(_root, 'notebooks'))  # cwd=notebooks/ so relative data paths resolve as before


In [ ]:
# ══════════════════════════════════════════════════════════════
# Configuration
# ══════════════════════════════════════════════════════════════
k       = 3      # k-point cumulant (number of external legs)
max_ell = 1      # maximum loop order (0 = tree only, 1 = tree + 1-loop, ...)

# External fields: one per external leg (must have length k).
# Pick from the physical fields of the model: ('dn', pop), ('dv', pop)
# where pop is a population index (1, 2, ...).
# Examples:
#   k=1:  [('dn', 1)]
#   k=2:  [('dn', 1), ('dn', 2)]
#   k=2:  [('dn', 1), ('dv', 1)]    (mixed correlator)
external_fields = [('dn', 1), ('dn', 1), ('dn', 2)]

USE_CACHE = True  # True: pull from cache when available; False: recompute all

assert len(external_fields) == k, f'external_fields has {len(external_fields)} entries but k={k}'
print(f'k = {k},  max loop order = {max_ell}')
print(f'External fields: {external_fields}')

---
## 1. Theory / Model Information

In [ ]:
m = HAWKES_MODEL
print(f"Model:  {m['name']}")
print(f"Convention: {m['background_rate_convention']}")
print()

print('── Index sets ──')
for name, vals in m['index_sets'].items():
    print(f'  {name}: {list(vals)}')
print()

print('── Response fields (not expanded — full integration variables) ──')
for f in m['response_fields']:
    print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
print()

print('── Physical fields (expanded around MF background) ──')
for f in m['physical_fields']:
    print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
print()

print('── Parameters ──')
for p in m.get('parameters', []):
    idx = '(indexed)' if p.get('indexed') else '(scalar)'
    dom = f", domain={p['domain']}" if p.get('domain') else ''
    print(f"  {p['name']}  {idx}{dom}  — {p.get('description', '')}")
print()

print('── Kernels ──')
for kern in m.get('kernels', []):
    print(f"  {kern['name']}  — {kern.get('description', '')}")
print()

print('── Operators ──')
for o in m.get('operators', []):
    print(f"  {o['name']}  (latex: ${o.get('latex_name', o['name'])}$)  — {o.get('description', '')}")
print()

print('── Nonlinear functions ──')
for f in m.get('functions', []):
    print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
print()

print('── Specializations ──')
print('  φ quadratic (cubic and higher coefficients = 0)')
print('  g = δ(t)  (instantaneous synaptic coupling)')

### 1.1 Expand the action and show all sectors

In [3]:
ft = FieldTheory(HAWKES_MODEL, taylor_order=4)
ft.expand()

R  = ft.ring()
ns = ft._ns

print(f'Taylor order: {ft.taylor_order}')
print(f'Polynomial ring: {R}')
print(f'Ring generators: {R.gens()}')
print(f'Response generators (n_tilde={ft._n_tilde}): {R.gens()[:ft._n_tilde]}')
print(f'Physical generators: {R.gens()[ft._n_tilde:]}')
print()

passed = ft.sanity_check()
print()
ft.summary()

Taylor order: 4
Polynomial ring: Multivariate Polynomial Ring in nt1, nt2, vt1, vt2, dn1, dn2, dv1, dv2 over Symbolic Ring
Ring generators: (nt1, nt2, vt1, vt2, dn1, dn2, dv1, dv2)
Response generators (n_tilde=4): (nt1, nt2, vt1, vt2)
Physical generators: (dn1, dn2, dv1, dv2)

=== Sanity checks ===
  [PASS]  (n_tilde=0, n_phys=0)  constant term
  [PASS]  (n_tilde=1, n_phys=0)  tadpole — must vanish at MF saddle
  [PASS]  (n_tilde=0, n_phys=1)  linear physical-only — must vanish at EOM

=== Action sectors ===
  (n_tilde=1, n_phys=1)  [free action]:


<IPython.core.display.Math object>

  (n_tilde=1, n_phys=2)  [vertex (order 3)]:


<IPython.core.display.Math object>

  (n_tilde=2, n_phys=0)  [noise kernel]:


...

<IPython.core.display.Math object>

  (n_tilde=4, n_phys=1)  [vertex (order 5)]:


<IPython.core.display.Math object>

  (n_tilde=4, n_phys=2)  [vertex (order 6)]:


<IPython.core.display.Math object>

### 1.2 Propagator

Extract the kernel matrix $\mathbf{K}$ from the free action, Fourier transform, and invert.

In [ ]:
import signal

S_free = ft.free_action()
ring_gen_names = [str(g) for g in R.gens()]

# Use ring variable ordering (must match build_field_index_map)
resp_names = ring_gen_names[:ft._n_tilde]
phys_names = ring_gen_names[ft._n_tilde:]

pos_to_row = {ring_gen_names.index(nm): i for i, nm in enumerate(resp_names)}
pos_to_col = {ring_gen_names.index(nm): j for j, nm in enumerate(phys_names)}

nf = len(resp_names)
K_data = [[SR(0)] * nf for _ in range(nf)]
for exp_vec, coeff in S_free.dict().items():
    row = col = None
    for idx in range(len(ring_gen_names)):
        if exp_vec[idx] > 0:
            if idx in pos_to_row: row = pos_to_row[idx]
            if idx in pos_to_col: col = pos_to_col[idx]
    if row is not None and col is not None:
        K_data[row][col] += SR(coeff)

K_mat = matrix(SR, K_data)

# Convert to kernel form
Dt       = ns.Dt
delta_D  = ns.delta_D
delta_Dp = ns.delta_Dp

def _to_kernel(c):
    c = SR(c)
    if c.has(delta_D) or c.has(ns.g):
        return c
    p0   = c.subs({Dt: 0})
    rest = (c - p0).subs({Dt: delta_Dp})
    return p0 * delta_D + rest

K_ker = matrix(SR, [[_to_kernel(K_mat[i, j]) for j in range(nf)] for i in range(nf)])

resp_sr  = [ns.vt[i] for i in ns.pop] + [ns.nt[i] for i in ns.pop]
phys_sr  = [ns.dv[i] for i in ns.pop] + [ns.dn[i] for i in ns.pop]

print('Field ordering:')
display(Math(r'\tilde{\mathbf{a}} = ' + latex(vector(resp_sr))
             + r',\qquad \mathbf{a} = ' + latex(vector(phys_sr))))
print()
print('Kernel matrix K (time domain):')
display(Math(r'\mathbf{K} = ' + latex(K_ker)))

# Fourier transform
t_var = SR.var('t')
omega = SR.var('omega', latex_name=r'\omega')

time_domain = {
    delta_D:  dirac_delta(t_var),
    delta_Dp: diff(dirac_delta(t_var), t_var),
    ns.g:     function('g')(t_var),
}

K_ft_data = [[SR(0)] * nf for _ in range(nf)]
for i in range(nf):
    for j in range(nf):
        c = K_ker[i, j]
        if not c.is_zero():
            K_ft_data[i][j] = fourier_transform(SR(c).subs(time_domain), t_var, omega)

K_ft = matrix(SR, K_ft_data)
print('Fourier-domain kernel:')
display(Math(r'\hat{\mathbf{K}}(\omega) = ' + latex(K_ft)))

# Propagator inverse
G_ft = K_ft.inverse().apply_map(lambda e: e.factor())
G_ft_explicit = True
print('Propagator:')
display(Math(r'\hat{\mathbf{G}}(\omega) = ' + latex(G_ft)))

# Adjugate, determinant, poles
adj_ft  = K_ft.adjugate()
D_omega = K_ft.det().expand()
D_prime = diff(D_omega, omega)

pole_eqs = solve(D_omega == 0, omega)
pole_vals = [eq.rhs() if hasattr(eq, 'rhs') else eq for eq in pole_eqs]

print(f'\ndet(K̂) = {D_omega}')
print(f'Poles ({len(pole_vals)}):')
for pk, p in enumerate(pole_vals):
    display(Math(r'\omega_{' + str(pk+1) + '} = ' + latex(p)))

# Residue matrices
C_mats = []
for omega_k in pole_vals:
    C_data = [[SR(0)] * nf for _ in range(nf)]
    for i in range(nf):
        for j in range(nf):
            n_ij = adj_ft[i, j]
            if not n_ij.is_zero():
                num = n_ij.subs({omega: omega_k})
                den = D_prime.subs({omega: omega_k})
                C_data[i][j] = (I * num / den).factor()
    C_mats.append(matrix(SR, C_data))

print('Residue matrices:')
for pk, C in enumerate(C_mats):
    display(Math(r'\mathbf{C}_{' + str(pk+1) + r'} = ' + latex(C)))

# Time-domain propagator
G_t = sum(C_mats[pk] * exp(I * pole_vals[pk] * t_var) for pk in range(len(pole_vals)))
G_t = G_t.apply_map(lambda e: e.simplify_full())
print('Time-domain propagator G(t) for t > 0:')
display(Math(r'\mathbf{G}(t) = ' + latex(G_t)))

---
## 2. Prediagram Enumeration

Enumerate all trees, topologies, and prediagrams for the $k$-point function
at each loop order $\ell = 0, \ldots, \ell_{\max}$.

In [5]:
from sage.plot.plot import graphics_array

def plot_prediagrams(pds, title_prefix=''):
    """Plot prediagrams with colored vertices."""
    if not pds:
        print('  (none)')
        return
    plots = []
    for i, (D, G, leaves, internal) in enumerate(pds):
        leaves_set = set(leaves)
        color_map = {}
        for v in D.vertices():
            if v in leaves_set:
                color_map.setdefault('black', []).append(v)
            elif D.in_degree(v) == 0:
                color_map.setdefault('red', []).append(v)
            else:
                color_map.setdefault('lightblue', []).append(v)
        plots.append(D.plot(vertex_colors=color_map, vertex_size=400,
                            edge_thickness=2, title=f"{title_prefix}{i+1}"))
    n_cols = min(4, len(plots))
    n_rows = (len(plots) + n_cols - 1) // n_cols
    graphics_array(plots, n_rows, n_cols).show(figsize=[5*n_cols, 4*n_rows])

def plot_topologies(topos, title_prefix=''):
    """Plot undirected topologies."""
    if not topos:
        print('  (none)')
        return
    plots = []
    for i, (G, leaves, internal) in enumerate(topos):
        leaves_set = set(leaves)
        color_map = {}
        for v in G.vertices():
            if v in leaves_set:
                color_map.setdefault('black', []).append(v)
            else:
                color_map.setdefault('lightgray', []).append(v)
        plots.append(G.plot(vertex_colors=color_map, vertex_size=400,
                            edge_thickness=2, title=f"{title_prefix}{i+1}"))
    n_cols = min(4, len(plots))
    n_rows = (len(plots) + n_cols - 1) // n_cols
    graphics_array(plots, n_rows, n_cols).show(figsize=[5*n_cols, 4*n_rows])

print('Plot helpers defined.')

Plot helpers defined.


In [ ]:
# ── Pipeline cache (depends on model + k) ──────────────────────
# Model name + external fields sanitized for filesystem use
_model_tag = HAWKES_MODEL['name'].replace(' ', '_').lower()
_ext_tag = '_'.join(f'{f[0]}{f[1]}' for f in external_fields)
cache_dir = f'saved_results/{_model_tag}_k{k}_{_ext_tag}'
cache = PipelineCache(cache_dir)
if USE_CACHE:
    print(f'Cache: {cache}')
    for e in cache.list_cached():
        print(f'  {e["key"]}  (saved {e["saved_at"][:19]})')
else:
    print('Cache disabled — all stages will be recomputed.')

# ── Enumerate prediagrams for each loop order ──────────────────
pds_by_ell = {}    # ell -> list of prediagrams
counts_by_ell = {} # ell -> counts dict

for ell in range(max_ell + 1):
    def _enumerate(ell=ell):
        t, tp, pd, c = enumerate_all(k, ell, verbose=False)
        return (pd, c)

    pds, counts = cache.get_or_compute(
        'prediagrams', _enumerate, k=k, loop_order=ell
    ) if USE_CACHE else _enumerate()

    pds_by_ell[ell] = pds
    counts_by_ell[ell] = counts

    print(f'ell={ell}:  {counts["n_trees"]} trees,  '
          f'{counts["n_topologies"]} topologies,  '
          f'{counts["n_prediagrams"]} prediagrams')
    plot_prediagrams(pds, title_prefix=f'ell={ell} PD ')

---
## 3. Vertex Extraction

Extract typed vertices (`VertexType`, `SourceType`) from the expanded action, based on the maximum vertex degree required by the prediagrams.

In [ ]:
from msrjd.enumeration.degree_scan import max_vertex_degree, scan_source_vertices

all_pds = [pd for pds in pds_by_ell.values() for pd in pds]
max_deg = max_vertex_degree(all_pds)
src_degs = scan_source_vertices(all_pds)
print(f'Max vertex degree across all prediagrams: {max_deg}')
print(f'Source vertex out-degrees needed: {src_degs}')
print(f'Current Taylor order: {ft.taylor_order}')
print(f'Sufficient: {ft.taylor_order >= max_deg}')
print()

vtypes = extract_vertex_types(ft)
stypes = extract_source_types(ft)

print(f'── Interaction vertex types ({len(vtypes)}) ──')
for i, vt in enumerate(vtypes):
    print(f'  V{i+1}: bigrade={vt.bigrade}, in_deg={vt.in_degree}, out_deg={vt.out_degree}')
    print(f'        resp_legs={vt.response_legs}, phys_legs={vt.physical_legs}')
    display(Math(f'\\quad\\text{{coeff}} = {latex(vt.coefficient)}'))

print(f'\n── Source types ({len(stypes)}) ──')
for i, st in enumerate(stypes):
    print(f'  S{i+1}: bigrade={st.bigrade}, out_deg={st.out_degree}')
    print(f'        resp_legs={st.response_legs}')
    display(Math(f'\\quad\\text{{coeff}} = {latex(st.coefficient)}'))

int_degs, src_degs_avail = available_degrees(vtypes, stypes)
print(f'\nAvailable interaction degrees (in, out): {sorted(int_degs)}')
print(f'Available source out-degrees: {sorted(src_degs_avail)}')

---
## 4. Prediagram Filtering

Remove prediagrams whose vertex degree signatures don't match any available vertex or source type.

In [ ]:
kept_by_ell = {}  # ell -> list of kept prediagrams

for ell in range(max_ell + 1):
    def _filter(ell=ell):
        kept, disc = filter_prediagrams(pds_by_ell[ell], vtypes, stypes)
        return (kept, disc)

    kept, disc = cache.get_or_compute(
        'filtered', _filter, k=k, loop_order=ell
    ) if USE_CACHE else _filter()

    kept_by_ell[ell] = kept
    print(f'ell={ell}:  {len(pds_by_ell[ell])} prediagrams -> '
          f'{len(kept)} kept, {disc} discarded')
    plot_prediagrams(kept, title_prefix=f'ell={ell} PD ')

---
## 5. Unique Typed Diagrams

For each surviving prediagram, enumerate all valid field-type assignments,
filter for causality, and deduplicate by isomorphism to obtain the set of
**unique Feynman diagrams** $\Gamma$.  Only these are cached — intermediate
typed diagrams are not stored.

In [ ]:
# External fields are specified in the Configuration cell above.
print(f'External fields (k={k}): {external_fields}')

# Build field index maps
ring_var_names_list = list(ns._ring_var_names)
n_tilde = ft._n_tilde
resp_idx, phys_idx = build_field_index_map(ring_var_names_list, n_tilde)

# Validate that each external field exists in the physical field index
for field in external_fields:
    assert field in phys_idx, (
        f'External field {field} not found in physical fields. '
        f'Available: {sorted(phys_idx.keys())}'
    )

print('\nResponse field index map:')
for leg, idx in sorted(resp_idx.items(), key=lambda x: x[1]):
    print(f'  {leg} -> row {idx}')
print()
print('Physical field index map:')
for leg, idx in sorted(phys_idx.items(), key=lambda x: x[1]):
    print(f'  {leg} -> col {idx}')

In [ ]:
unique_by_ell = {}  # ell -> list of unique TypedDiagram

for ell in range(max_ell + 1):
    def _unique(ell=ell):
        typed = enumerate_all_typed(
            kept_by_ell[ell], external_fields, vtypes, stypes,
            G_ft, resp_idx, phys_idx
        )
        causal, n_disc, _ = filter_causal(typed)
        unique = deduplicate_typed_diagrams(causal)
        print(f'  ell={ell}: {len(kept_by_ell[ell])} prediagrams -> {len(typed)} typed'
              f' -> {len(causal)} causal -> {len(unique)} unique')
        return unique

    unique_by_ell[ell] = cache.get_or_compute(
        'unique_typed', _unique, k=k, loop_order=ell
    ) if USE_CACHE else _unique()

    print(f'ell={ell}: {len(unique_by_ell[ell])} unique diagrams')

# Convenience aliases used downstream
unique_tree = unique_by_ell.get(0, [])
all_unique = [td for ell in range(max_ell + 1) for td in unique_by_ell.get(ell, [])]
print(f'\nTotal unique diagrams across all loop orders: {len(all_unique)}')

---
## 6. Diagram Details & Combinatorial Factor $M(\Gamma)$

For each unique diagram $\Gamma$, display the vertex assignments, edge types,
and the **combinatorial factor** $M(\Gamma)$ together with the scalar prefactor.

The vertex coefficients shown include the $(-1)$ from the partition function
convention $Z = \int \exp(-S)$.

In [ ]:
from msrjd.diagrams.symmetry import classify_coefficient_factors

# Get time-dependence metadata from the model
time_dep_params = HAWKES_MODEL.get('time_dependent_parameters', [])
noise_structure = HAWKES_MODEL.get('noise_structure', {
    'temporal_type': 'white', 'amplitude_params': []
})
print(f'Noise temporal type: {noise_structure.get("temporal_type", "white")}')
print(f'Time-dependent params: {time_dep_params if time_dep_params else "(none -- stationary)"}')
print()


def show_unique_diagram(td, idx, winfo):
    """Display a unique diagram with M(Gamma) and weight structure."""
    M = winfo['M']
    print(f'\n{"="*60}')
    print(f'Unique Diagram {idx}    M = {M}')
    print(f'{"="*60}')

    print('  External legs:')
    for lf, field in sorted(td.external_legs.items()):
        print(f'    leaf {lf} <- {field[0]}{field[1]}')

    if td.vertex_assignments:
        print('  Vertex assignments:')
        for v, vtype in sorted(td.vertex_assignments.items()):
            tname = type(vtype).__name__
            print(f'    v{v} ({tname}): bigrade={vtype.bigrade}, '
                  f'resp={vtype.response_legs}', end='')
            if hasattr(vtype, 'physical_legs'):
                print(f', phys={vtype.physical_legs}', end='')
            print()
            eff_coeff = -SR(vtype.coefficient)
            display(Math(f'\\quad (-1)\\cdot c_{{v_{v}}} = {latex(eff_coeff)}'))

    print('  Edges (propagator assignments):')
    for edge_key in sorted(td.edge_types.keys()):
        resp_leg, phys_leg = td.edge_types[edge_key]
        u, v = edge_key[0], edge_key[1]
        ri, pi = td.propagator_indices.get(edge_key, ('?', '?'))
        print(f'    {u} -> {v}:  {resp_leg[0]}{resp_leg[1]} -> '
              f'{phys_leg[0]}{phys_leg[1]}  [G_hat[{ri},{pi}]]')

    sp = winfo['scalar_prefactor']
    display(Math(
        r'\text{Scalar prefactor: }\;' + latex(sp)
    ))


# ── Display all unique diagrams by loop order ──
for ell in range(max_ell + 1):
    diagrams = unique_by_ell.get(ell, [])
    ell_label = 'TREE-LEVEL' if ell == 0 else f'{ell}-LOOP'
    print('='*60)
    print(f'{ell_label} UNIQUE DIAGRAMS ({len(diagrams)})')
    print('='*60)

    for i, td in enumerate(diagrams, 1):
        winfo = classify_coefficient_factors(td, time_dep_params, noise_structure)
        label = f'Tree-{i}' if ell == 0 else f'{ell}L-{i}'
        show_unique_diagram(td, label, winfo)

---
## 7. Symbolic Integration (Stationary Case)

For each unique diagram $\Gamma$, construct the **unevaluated frequency-domain
integral** that gives the time-domain contribution $C_\Gamma(t_1, t_2)$.

### Procedure

1. **Assign** a frequency variable $\omega_{e_i}$ to each directed edge.

2. **Solve conservation** at every internal vertex (interaction and source alike):
   $$\delta\!\Big(\sum_{\text{in}} \omega_e - \sum_{\text{out}} \omega_e\Big)$$
   This expresses internal edge frequencies in terms of external ones.
   If the diagram has $\ell$ loops, $\ell$ internal frequencies remain free.

3. **Build the integrand**: for each edge, substitute the resolved frequency
   into the propagator entry $\hat{G}_{i,j}(\omega_e)$.  Each external leg
   contributes an exponential $e^{\pm i\omega t}$ from the inverse Fourier
   transform (sign from leaf directionality: tail $\to e^{-i\omega t}$,
   head $\to e^{+i\omega t}$).

4. **Display** the full unevaluated integral:
   $$C_\Gamma(t_1, t_2) = \text{scalar\_pf}
   \;\prod_j \int\!\frac{d\omega_j}{2\pi}\;
   \Bigl[\prod_e \hat{G}_{i_e,j_e}(\omega_e)\Bigr]
   \Bigl[\prod_{\text{ext}} e^{\pm i\omega t}\Bigr]$$

In [ ]:
from msrjd.integration.symbolic import (
    check_propagator_available,
    build_integrand_stationary,
    extract_propagator_factors,
    canonicalize_prop_factors,
    loop_kernel_signature,
    group_diagrams_by_kernel,
)

# ── Propagator data dict (assembled from Section 1.2) ──
omega = SR.var('omega', latex_name=r'\omega')

propagator_data = {
    'G_ft': G_ft,
    'adj_ft': adj_ft,
    'D_omega': D_omega,
    'G_ft_explicit': True,
    'pole_vals': pole_vals,
    'C_mats': C_mats,
    'nf': nf,
}

prop_mode = check_propagator_available(propagator_data)
print(f'Propagator mode: {prop_mode}')
print(f'Poles: {pole_vals}')
print()


def _diagram_label(td):
    """Return a human-readable label like 'Tree-3' or '1L-17'."""
    for ell in range(max_ell + 1):
        diagrams = unique_by_ell.get(ell, [])
        if td in diagrams:
            idx = diagrams.index(td) + 1
            return f'Tree-{idx}' if ell == 0 else f'{ell}L-{idx}'
    return '??'


def show_integral(td, label, propagator_data, omega_sym):
    """
    Build and display the unevaluated integral for a typed diagram.
    Returns the integrand result dict.
    """
    ir = build_integrand_stationary(
        td, propagator_data, k=k,
        omega_symbol=omega_sym,
        time_dep_params=HAWKES_MODEL.get('time_dependent_parameters', []),
        noise_structure=HAWKES_MODEL.get('noise_structure'),
    )

    prefactor = ir['scalar_prefactor']
    full_integrand = ir['full_integrand']
    ext_freqs = ir['ext_freqs']
    free_freqs = ir['free_freqs']
    ell = ir['loop_number']
    overall = ir['overall_conservation']

    print(f'\n{"="*60}')
    print(f'{label}   (ell = {ell})')
    print(f'{"="*60}')

    print(f'  Free (loop) frequencies: {free_freqs if free_freqs else "(none)"}')
    print(f'  Independent external frequencies: {ext_freqs}')

    if overall is not None:
        display(Math(
            r'  \text{Overall conservation: }\;'
            + latex(overall) + r' = 0'
        ))

    int_vars = list(free_freqs) + list(ext_freqs)
    integrals_tex = ''
    for v in int_vars:
        integrals_tex += r'\int\!\frac{d' + latex(v) + r'}{2\pi}\;'

    pf_tex = latex(prefactor)
    integrand_tex = latex(full_integrand)

    display(Math(
        r'C_{\Gamma}(t_1, t_2) = '
        + pf_tex + r'\;'
        + integrals_tex
        + integrand_tex
    ))

    integrand_only = ir['integrand']
    display(Math(
        r'\text{Propagator product: }\;'
        + latex(integrand_only)
    ))

    exp_factor = (full_integrand / integrand_only)
    try:
        exp_factor = exp_factor.simplify()
    except Exception:
        pass
    display(Math(
        r'\text{Exponential factor: }\;'
        + latex(exp_factor)
    ))

    return ir


# ═══════════════════════════════════════════════════════════════
# Show integrals for each loop order
# ═══════════════════════════════════════════════════════════════
ir_by_ell = {}  # ell -> list of integrand results

for ell in range(max_ell + 1):
    diagrams = unique_by_ell.get(ell, [])
    ell_label = 'TREE-LEVEL' if ell == 0 else f'{ell}-LOOP'

    print('='*60)
    print(f'{ell_label} INTEGRALS ({len(diagrams)})')
    print('='*60)

    ir_list = []
    for i, td in enumerate(diagrams, 1):
        label = f'Tree-{i}' if ell == 0 else f'{ell}L-{i}'
        ir_i = show_integral(td, label, propagator_data, omega)
        ir_list.append(ir_i)
    ir_by_ell[ell] = ir_list

### 7.2 Unique loop kernels

Group all diagrams by their **loop kernel** -- the product of propagator entries
$\prod_e \hat{G}_{i_e,j_e}(\omega_e)$ with the same frequency routing.

Diagrams sharing a loop kernel differ only in their scalar prefactors (vertex
coefficients, combinatorial factors).  These prefactors simply add, so we only
need to numerically integrate each unique kernel once.

For each unique kernel, we show:
- Which diagrams it came from
- The combined scalar prefactor
- The propagator factors (opaque representation)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Group ALL diagrams by loop kernel
# ═══════════════════════════════════════════════════════════════
from msrjd.integration.symbolic import _factor_depends_on

kernel_groups = group_diagrams_by_kernel(
    all_unique, propagator_data, k=k,
    omega_symbol=omega,
    time_dep_params=HAWKES_MODEL.get('time_dependent_parameters', []),
    noise_structure=HAWKES_MODEL.get('noise_structure'),
)

print(f'Total unique diagrams: {len(all_unique)}')
print(f'Unique loop kernels:   {len(kernel_groups)}')
print(f'Numerical integrations saved: {len(all_unique) - len(kernel_groups)}')
print()


def _factor_to_latex(f):
    """Render a single opaque propagator factor as LaTeX."""
    if f[0] == 'prop':
        _, ri, pi, omega_expr = f
        return (r'\hat{G}_{' + str(ri) + ',' + str(pi) + r'}('
                + latex(omega_expr) + ')')
    elif f[0] == 'noise':
        return r'\hat{\kappa}(' + latex(f[1]) + ')'
    return str(f)


for g_idx, g in enumerate(kernel_groups, 1):
    ell = g['loop_number']
    n = g['n_diagrams']
    combined_pf = g['combined_prefactor']
    ir = g['representative_ir']

    print(f'{"="*60}')
    print(f'Kernel {g_idx}   (ell = {ell}, {n} diagram{"s" if n > 1 else ""})')
    print(f'{"="*60}')

    for j, td in enumerate(g['diagrams']):
        pf_j = g['individual_prefactors'][j]
        print(f'  {_diagram_label(td)}:  prefactor = {pf_j}')

    display(Math(
        r'\text{Combined prefactor: }\;'
        + latex(combined_pf)
    ))

    pf_list = g['prop_factors']
    loop_vars_canonical = [SR.var(f'L_{i}') for i in range(ell)]

    if ell > 0:
        loop_factors = [f for f in pf_list
                        if _factor_depends_on(f, loop_vars_canonical)]
        ext_factors = [f for f in pf_list
                       if not _factor_depends_on(f, loop_vars_canonical)]
    else:
        loop_factors = []
        ext_factors = list(pf_list)

    ext_tex = (r' \cdot '.join(_factor_to_latex(f) for f in ext_factors)
               if ext_factors else '')
    loop_tex = (r' \cdot '.join(_factor_to_latex(f) for f in loop_factors)
                if loop_factors else '')

    if ell == 0:
        if ext_tex:
            display(Math(r'\text{Propagator product: }\;' + ext_tex))
    else:
        loop_var_tex = r' \cdot '.join(
            r'\frac{d' + latex(v) + r'}{2\pi}'
            for v in loop_vars_canonical
        )
        integral_tex = r'\int_{-\infty}^{\infty} ' + loop_var_tex
        if loop_tex:
            integral_tex += r'\; ' + loop_tex

        if ext_tex:
            full_tex = ext_tex + r' \;\times\; ' + integral_tex
        else:
            full_tex = integral_tex

        display(Math(r'\text{Loop integral: }\;' + full_tex))

        loop_vars_tex = ', '.join(latex(v) for v in ir['free_freqs'])
        ext_vars_tex = ', '.join(latex(v) for v in ir['ext_freqs'])
        print(f'  Loop variables: {{{loop_vars_tex}}}')
        print(f'  External variables: {{{ext_vars_tex}}}')

    print()

### 7.3 Unique loop integrands

Section 7.2 groups diagrams by their **full** kernel signature
(external factors + loop integrand).  But for numerical integration
we only need to evaluate each **distinct loop integrand** once --
diagrams that share the same loop integrand but differ in their
external propagator factors still require only one numerical
integration.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7.3  Unique loop integrands
# ═══════════════════════════════════════════════════════════════
from collections import defaultdict as _defaultdict
from msrjd.integration.symbolic import loop_only_signature

# ── Group kernel_groups by their loop-only signature ─────────
loop_integrand_groups = _defaultdict(list)  # loop_sig -> list of kernel group dicts

for g in kernel_groups:
    if g['loop_number'] == 0:
        continue
    lsig = loop_only_signature(g['signature'])
    loop_integrand_groups[lsig].append(g)

n_tree = sum(1 for g in kernel_groups if g['loop_number'] == 0)
n_full_loop_kernels = sum(1 for g in kernel_groups if g['loop_number'] > 0)
n_unique_loop_integrands = len(loop_integrand_groups)
n_loop_diagrams = sum(
    g['n_diagrams'] for g in kernel_groups if g['loop_number'] > 0
)

print(f'Tree-level kernels (no integration needed):      {n_tree}')
print(f'Full loop kernels (ext + loop signature):        {n_full_loop_kernels}')
print(f'Unique loop integrands (loop signature only):    {n_unique_loop_integrands}')
print(f'Total loop diagrams covered:                     {n_loop_diagrams}')
print(f'Numerical integrations saved:                    '
      f'{n_loop_diagrams - n_unique_loop_integrands}')
print()

for li, (lsig, groups) in enumerate(loop_integrand_groups.items(), 1):
    ell = groups[0]['loop_number']
    all_labels = []
    total_diagrams = 0
    for g in groups:
        total_diagrams += g['n_diagrams']
        for td in g['diagrams']:
            all_labels.append(_diagram_label(td))

    pf_list = groups[0]['prop_factors']
    loop_vars_canonical = [SR.var(f'L_{i}') for i in range(ell)]

    lf = [f for f in pf_list if _factor_depends_on(f, loop_vars_canonical)]

    loop_var_tex = r' \cdot '.join(
        r'\frac{d' + latex(v) + r'}{2\pi}'
        for v in loop_vars_canonical
    )
    integrand_body = r' \cdot '.join(_factor_to_latex(f) for f in lf) if lf else '1'
    integrand_tex = (r'\int_{-\infty}^{\infty} '
                     + loop_var_tex + r'\; ' + integrand_body)

    label_str = ', '.join(all_labels)
    if len(all_labels) > 6:
        label_str = ', '.join(all_labels[:6]) + f', ... ({total_diagrams} total)'

    print(f'── Loop integrand {li}/{n_unique_loop_integrands}  '
          f'(ell={ell}, {total_diagrams} diagram{"s" if total_diagrams>1 else ""}: '
          + label_str + ')')
    display(Math(integrand_tex))
    print()

### 7.4 Numerical evaluation -- spectral grid + inverse FFT

For each unique loop kernel, numerically evaluate the loop integral
on a spectral grid, then combine with the tree-level contributions
and IFFT to the time domain.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7.4  Numerical evaluation — adaptive by k
# ═══════════════════════════════════════════════════════════════
import numpy as np
from scipy.optimize import fsolve
from sage.all import numerical_integral, CC, fast_callable, CDF, diff

# ═══════════════════════════════════════════════════════════════
# Fundamental parameters (model-agnostic: user specifies these)
# ═══════════════════════════════════════════════════════════════
fundamental = {
    'E':   [1.0, 0.5],                        # external drive per population
    'w':   [[0.5, 0.25], [0.4, 0.3]],         # synaptic weights w_ij
    'tau': 10.0,                               # membrane time constant
    'a':   [0.5, 0.4],                         # quadratic gain: phi_i(v) = (a_i/2) v^2
}

npop = len(ns.pop)

print('Fundamental parameters:')
for key, val in fundamental.items():
    print(f'  {key} = {val}')

# ═══════════════════════════════════════════════════════════════
# Solve mean-field equations from the model
# ═══════════════════════════════════════════════════════════════
# Read phi_concrete from model and differentiate symbolically
v_sym = SR.var('_v_mf_')
taylor_order = ft.taylor_order

# Build substitution dict: map model parameter symbols to fundamental values.
# This handles any model (linear phi has no 'a', quadratic has 'a', etc.)
_param_subs = {}
for pspec in HAWKES_MODEL.get('parameters', []):
    pname = pspec['name']
    if pname in fundamental:
        if pspec.get('indexed', False):
            for i in ns.pop:
                sym = getattr(ns, pname)[i]
                _param_subs[sym] = fundamental[pname][i]
        else:
            sym = getattr(ns, pname)
            _param_subs[sym] = fundamental[pname]

phi_derivs = {}  # phi_derivs[i][dk] = k-th derivative of phi_i(v), symbolic
for i in ns.pop:
    phi_expr = HAWKES_MODEL['phi_concrete'](ns, i, v_sym)
    phi_derivs[i] = {}
    for dk in range(taylor_order + 1):
        if dk == 0:
            phi_derivs[i][dk] = phi_expr
        else:
            phi_derivs[i][dk] = diff(phi_expr, v_sym, dk)

# Build numerical phi callable from the symbolic expression
def phi_num(i, v_val):
    """Evaluate phi_i(v) numerically."""
    return float(phi_derivs[i][0].subs(_param_subs).subs({v_sym: v_val}))

# Solve MF self-consistency: n*_i = phi_i(E_i + sum_j w_ij * n*_j)
def mf_residual(nstar_vec):
    residuals = []
    for i in ns.pop:
        v_star_i = fundamental['E'][i] + sum(
            fundamental['w'][i][j] * nstar_vec[j] for j in ns.pop
        )
        residuals.append(nstar_vec[i] - phi_num(i, v_star_i))
    return residuals

# Initial guess: small positive rates
nstar_guess = [1.0] * npop
nstar_sol = fsolve(mf_residual, nstar_guess, full_output=True)
nstar_vals = [float(x) for x in nstar_sol[0]]
info = nstar_sol[1]

# Compute v* and phi derivatives at the fixed point
vstar_vals = []
phi_deriv_vals = {}  # phi_deriv_vals[(dk, i)] = d^k phi_i / dv^k |_{v=v*}
for i in ns.pop:
    v_star_i = fundamental['E'][i] + sum(
        fundamental['w'][i][j] * nstar_vals[j] for j in ns.pop
    )
    vstar_vals.append(v_star_i)
    for dk in range(taylor_order + 1):
        phi_deriv_vals[(dk, i)] = float(
            phi_derivs[i][dk].subs(_param_subs).subs({v_sym: v_star_i})
        )

# Sanity check: phi_i(v*_i) should equal n*_i
print(f'\nMean-field solution:')
for i in ns.pop:
    phi_at_vstar = phi_deriv_vals[(0, i)]
    print(f'  pop {i+1}:  v* = {vstar_vals[i]:.6f},  '
          f'n* = {nstar_vals[i]:.6f},  '
          f'phi(v*) = {phi_at_vstar:.6f}  '
          f'{"OK" if abs(phi_at_vstar - nstar_vals[i]) < 1e-10 else "MISMATCH!"}')

print(f'\nDerived phi derivatives:')
for i in ns.pop:
    derivs_str = ', '.join(
        f"phi{dk}_{i+1} = {phi_deriv_vals[(dk, i)]:.6f}"
        for dk in range(1, taylor_order + 1)
        if (dk, i) in phi_deriv_vals
    )
    print(f'  pop {i+1}:  {derivs_str}')

# ═══════════════════════════════════════════════════════════════
# Assemble num_params for symbolic substitution
# ═══════════════════════════════════════════════════════════════
num_params = {}

# Weights
for i in ns.pop:
    for j in ns.pop:
        num_params[SR.var(f'w{i+1}{j+1}')] = fundamental['w'][i][j]

# Timescale
num_params[SR.var('tau')] = fundamental['tau']

# Derived: nstar
for i in ns.pop:
    num_params[ns.nstar[i]] = float(nstar_vals[i])

# Derived: phi derivatives (phi1, phi2, ..., up to taylor_order)
for i in ns.pop:
    for dk in range(1, taylor_order + 1):
        sym = SR.var(f'phi{dk}_{i+1}')
        if (dk, i) in phi_deriv_vals:
            num_params[sym] = phi_deriv_vals[(dk, i)]

print(f'\nFull num_params:')
for sym, val in num_params.items():
    print(f'  {sym} = {val}')

# Check pole stability
print(f'\nPoles (numerical):')
for p_idx, p in enumerate(pole_vals):
    p_num = complex(CC(p.subs(num_params)))
    print(f'  pole {p_idx+1}: {p_num:.6f}   (Im = {p_num.imag:.4f})')


# ═══════════════════════════════════════════════════════════════
# Evaluation helpers
# ═══════════════════════════════════════════════════════════════

def _find_integrand_vars(ir, num_params):
    """
    Substitute num_params into ir['integrand'] and identify which
    omega variables remain.  Returns (integrand_num, ext_vars, loop_vars).
    ext_vars/loop_vars are lists (possibly empty).
    """
    integrand_num = ir['integrand'].subs(num_params)
    remaining = sorted(integrand_num.variables(), key=str)

    ext_set = set(ir['ext_freqs'])
    free_set = set(ir['free_freqs'])

    ext_vars = [v for v in remaining if v in ext_set]
    loop_vars = [v for v in remaining if v in free_set]
    other = [v for v in remaining if v not in ext_set and v not in free_set]
    loop_vars += other  # anything unclassified goes to loop

    return integrand_num, ext_vars, loop_vars


def _scalar_loop_integral(ir, num_params, Omega_max=50, n_quad=1000):
    """
    Compute a pure scalar loop integral (no external frequencies).
    Returns a single complex number.
    """
    prefactor = float(ir['scalar_prefactor'].subs(num_params))
    integrand_num, ext_vars, loop_vars = _find_integrand_vars(ir, num_params)

    if not loop_vars:
        # No integration needed — just a constant (tree-level for k=1)
        return prefactor * complex(CDF(integrand_num))

    omega_loop = loop_vars[0]
    f_fast = fast_callable(integrand_num, vars=[omega_loop], domain=CDF)
    Omega_pts = np.linspace(-Omega_max, Omega_max, n_quad)
    dOmega = Omega_pts[1] - Omega_pts[0]
    vals = np.array([complex(f_fast(Om)) for Om in Omega_pts])
    result = np.trapz(vals, dx=dOmega) / (2 * np.pi)
    return prefactor * result


def spectrum_tree(ir, num_params, omega_grid):
    """Evaluate tree-level spectrum Chat on grid.
    
    For k=2 (1 ext freq): returns 1D array Chat(omega).
    For k>=3 (multiple ext freqs): returns nD array Chat(omega_0, omega_1, ...).
    """
    prefactor = float(ir['scalar_prefactor'].subs(num_params))
    integrand_num, ext_vars, loop_vars = _find_integrand_vars(ir, num_params)

    if not ext_vars:
        val = prefactor * complex(CDF(integrand_num))
        if isinstance(omega_grid, np.ndarray) and omega_grid.ndim == 1:
            return np.full(len(omega_grid), val, dtype=complex)
        return val

    n_ext = len(ext_vars)
    f_fast = fast_callable(integrand_num, vars=ext_vars, domain=CDF)

    if n_ext == 1:
        Chat = np.array([complex(f_fast(w)) for w in omega_grid])
    elif n_ext == 2:
        N = len(omega_grid)
        Chat = np.zeros((N, N), dtype=complex)
        for i, w0 in enumerate(omega_grid):
            for j, w1 in enumerate(omega_grid):
                Chat[i, j] = complex(f_fast(w0, w1))
    else:
        raise NotImplementedError(
            f'spectrum_tree: {n_ext} external frequencies not yet supported. '
            f'Need {n_ext}D grid evaluation.')

    return prefactor * Chat


def spectrum_one_loop(ir, num_params, omega_grid, Omega_max=50, n_quad=1000):
    """
    Evaluate one-loop spectrum Chat(omega) on a grid of external frequency.
    For each external omega, numerically integrate the loop frequency.
    """
    prefactor = float(ir['scalar_prefactor'].subs(num_params))
    integrand_num, ext_vars, loop_vars = _find_integrand_vars(ir, num_params)

    if not loop_vars:
        print('  1-loop: no loop variable found — treating as tree')
        return spectrum_tree(ir, num_params, omega_grid)

    omega_loop = loop_vars[0]

    if not ext_vars:
        # Pure loop integral, no external freq.  Return scalar broadcast.
        f_fast = fast_callable(integrand_num, vars=[omega_loop], domain=CDF)
        Omega_pts = np.linspace(-Omega_max, Omega_max, n_quad)
        dOmega = Omega_pts[1] - Omega_pts[0]
        vals = np.array([complex(f_fast(Om)) for Om in Omega_pts])
        scalar = prefactor * np.trapz(vals, dx=dOmega) / (2 * np.pi)
        return np.full(len(omega_grid), scalar, dtype=complex)

    ext_var = ext_vars[0]
    f_fast = fast_callable(integrand_num, vars=[ext_var, omega_loop], domain=CDF)
    Omega_pts = np.linspace(-Omega_max, Omega_max, n_quad)
    dOmega = Omega_pts[1] - Omega_pts[0]

    Chat = np.zeros(len(omega_grid), dtype=complex)
    for i, w_ext in enumerate(omega_grid):
        vals = np.array([complex(f_fast(w_ext, Om)) for Om in Omega_pts])
        Chat[i] = np.trapz(vals, dx=dOmega) / (2 * np.pi)

    return prefactor * Chat


def inverse_fourier(Chat, omega_grid):
    """Inverse DFT: C(tau) where tau = t1 - t2.
    
    The MSR-JD integrand encodes C(t1-t2) via exp(+iw(t1-t2)).
    numpy ifft with e^{+iwt} gives C(t1-t2) directly.
    
    Handles 1D (k=2) and 2D (k=3) spectra.
    """
    N = len(omega_grid)
    Delta_omega = omega_grid[1] - omega_grid[0]
    tau_grid = np.fft.fftshift(np.fft.fftfreq(N, d=Delta_omega / (2 * np.pi)))
    scale = N * Delta_omega / (2 * np.pi)

    if Chat.ndim == 1:
        Chat_shifted = np.fft.ifftshift(Chat)
        C_tau = np.fft.fftshift(np.fft.ifft(Chat_shifted)) * scale
    elif Chat.ndim == 2:
        Chat_shifted = np.fft.ifftshift(Chat, axes=(0, 1))
        C_tau = np.fft.fftshift(np.fft.ifft2(Chat_shifted)) * scale**2
    else:
        raise NotImplementedError(f'inverse_fourier: {Chat.ndim}D not supported')

    return tau_grid, C_tau


def evaluate_kernel_group(g, num_params, omega_grid=None,
                          Omega_max=50, n_quad=1000):
    """
    Evaluate a kernel group.
    - If omega_grid is None (k=1), computes a scalar loop integral.
    - Otherwise evaluates on the frequency grid.
    """
    ir = g['representative_ir']
    ell = g['loop_number']
    combined_pf = g['combined_prefactor']

    ir_eval = dict(ir)
    ir_eval['scalar_prefactor'] = combined_pf

    if omega_grid is None:
        # Scalar mode (k=1): just compute the loop integral
        return _scalar_loop_integral(ir_eval, num_params,
                                     Omega_max=Omega_max, n_quad=n_quad)

    if ell == 0:
        return spectrum_tree(ir_eval, num_params, omega_grid)
    else:
        return spectrum_one_loop(ir_eval, num_params, omega_grid,
                                 Omega_max=Omega_max, n_quad=n_quad)


# ═══════════════════════════════════════════════════════════════

def _build_factor_product(factors, propagator_data, omega_symbol, num_params):
    """Build a symbolic product from a list of canonical prop factors."""
    G_ft = propagator_data.get('G_ft')
    product = SR(1)
    for f in factors:
        if f[0] == 'prop':
            _, ri, pi, omega_expr = f
            if G_ft is not None:
                entry = SR(G_ft[ri, pi]).subs({omega_symbol: omega_expr})
            else:
                adj = propagator_data['adj_ft']
                det = propagator_data['D_omega']
                entry = (SR(adj[ri, pi]) / SR(det)).subs({omega_symbol: omega_expr})
            product *= entry
        elif f[0] == 'noise':
            pass  # white noise: no frequency-dependent factor
    return product.subs(num_params)


def _eval_product_on_grid(expr, ext_var, omega_grid):
    """Evaluate a symbolic expression on the omega grid."""
    if ext_var is None or not expr.has(ext_var):
        val = complex(CDF(expr))
        return np.full(len(omega_grid), val, dtype=complex)
    f_fast = fast_callable(expr, vars=[ext_var], domain=CDF)
    return np.array([complex(f_fast(w)) for w in omega_grid])


def _eval_loop_integral_on_grid(loop_integrand_expr, ext_var, loop_var,
                                omega_grid, num_params,
                                Omega_max=50, n_quad=1000):
    """
    Evaluate int loop_integrand(omega_ext, Omega) dOmega/(2pi)
    on the omega grid.  Returns array of shape (N_fft,).
    """
    if ext_var is not None and loop_var is not None:
        f_fast = fast_callable(loop_integrand_expr,
                               vars=[ext_var, loop_var], domain=CDF)
    elif loop_var is not None:
        f_fast = fast_callable(loop_integrand_expr,
                               vars=[loop_var], domain=CDF)
    else:
        # No loop variable — just a constant
        val = complex(CDF(loop_integrand_expr))
        return np.full(len(omega_grid), val, dtype=complex)

    Omega_pts = np.linspace(-Omega_max, Omega_max, n_quad)
    dOmega = Omega_pts[1] - Omega_pts[0]

    if ext_var is not None:
        result = np.zeros(len(omega_grid), dtype=complex)
        for i, w_ext in enumerate(omega_grid):
            vals = np.array([complex(f_fast(w_ext, Om)) for Om in Omega_pts])
            result[i] = np.trapz(vals, dx=dOmega) / (2 * np.pi)
        return result
    else:
        vals = np.array([complex(f_fast(Om)) for Om in Omega_pts])
        scalar = np.trapz(vals, dx=dOmega) / (2 * np.pi)
        return np.full(len(omega_grid), scalar, dtype=complex)


# Dispatch by k
# ═══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

print(f'\n{"="*60}')
print(f'k = {k}  —  Evaluating {len(kernel_groups)} unique kernel(s)')
print(f'{"="*60}')

if k == 1:
    # ───────────────────────────────────────────────────────────
    # k = 1 :  Mean = mean-field + loop corrections
    # ───────────────────────────────────────────────────────────
    # The mean-field value n* comes from solving the classical rate
    # equations (saddle point of the action) — not from a diagram.
    # Diagrams compute fluctuation corrections on top of n*:
    #   <dn> = n* + delta<dn>_{1-loop} + delta<dn>_{2-loop} + ...

    # Compute unique loop integrals, sum per loop level
    from collections import defaultdict as _ddict
    correction_by_ell = _ddict(complex)

    if loop_integrand_groups:
        n_unique = len(loop_integrand_groups)
        n_total = sum(sum(g['n_diagrams'] for g in gs)
                      for gs in loop_integrand_groups.values())
        print(f'\n{n_unique} unique loop integrand(s), {n_total} diagram(s)')

        for li, (lsig, groups) in enumerate(loop_integrand_groups.items(), 1):
            ell = groups[0]['loop_number']
            total_diags = sum(g['n_diagrams'] for g in groups)

            # Evaluate the loop integral once with unit prefactor
            representative = groups[0]
            ir_unit = dict(representative['representative_ir'])
            ir_unit['scalar_prefactor'] = SR(1)
            raw_val = _scalar_loop_integral(ir_unit, num_params)

            # Sum (combined_prefactor × loop_integral) across groups
            val = sum(
                float(g['combined_prefactor'].subs(num_params)) * raw_val
                for g in groups
            )
            correction_by_ell[ell] += val

            print(f'  Integrand {li}/{n_unique} (ell={ell}, '
                  f'{total_diags} diagrams): {val:.6e}')

    # Report
    field_name, pop_idx = external_fields[0]
    field_label = f'{field_name}_{pop_idx}'
    nstar_sym = SR.var(f'nstar{pop_idx}')
    nstar_val = float(num_params.get(nstar_sym, 0))

    print(f'\n{"="*60}')
    print(f'k=1 Mean: <{field_label}>')
    print(f'{"="*60}')
    print(f'  {"Mean-field (n*)":25s} = {nstar_val: .6e}')

    loop_total = 0.0
    for ell in sorted(correction_by_ell):
        val = correction_by_ell[ell]
        label = f'{ell}-loop correction'
        re = val.real
        loop_total += re
        im_warn = ''
        if abs(val.imag) > 1e-12 * max(abs(val.real), 1e-30):
            im_warn = f'  [Im = {val.imag:.4e}]'
        print(f'  {label:25s} = {re: .6e}{im_warn}')

    total = nstar_val + loop_total
    print(f'  {chr(9472)*40}')
    print(f'  {"Total <" + field_label + ">":25s} = {total: .6e}')
    print(f'  (Loop / MF ratio:  {abs(loop_total/nstar_val):.2e})')

    # Stationary k=1: each loop order is a scalar — printed table
    # is the natural output.  (Nonstationary k=1 will produce <n>(t)
    # curves per loop order once that code path is implemented.)

else:
    # ───────────────────────────────────────────────────────────
    # k >= 2 :  Frequency grid + inverse FFT
    # ───────────────────────────────────────────────────────────
    n_ext = k - 1  # number of independent ext frequencies (stationary)
    if n_ext <= 1:
        T_max = 80.0
        Delta_tau = 0.05
    else:
        # k>=3: high resolution for clean nD IFT
        T_max = 80.0
        Delta_tau = 0.02
    N_fft = int(2 * T_max / Delta_tau)
    N_fft = int(2**np.ceil(np.log2(N_fft)))
    Delta_omega = 2 * np.pi / (N_fft * Delta_tau)
    omega_max = N_fft * Delta_omega / 2
    omega_grid = np.arange(-N_fft//2, N_fft//2) * Delta_omega
    mid = N_fft // 2

    print(f'\nGrid: N={N_fft}, omega_max={omega_max:.2f}, '
          f'Delta_omega={Delta_omega:.4f}, Delta_tau={Delta_tau}')

    # ── Step 1: Tree-level (ell=0) — sum all tree spectra ──
    tree_groups = [g for g in kernel_groups if g['loop_number'] == 0]
    n_ext = k - 1  # number of independent external frequencies (stationary)

    if n_ext <= 1:
        # k<=2: evaluate on 1D grid directly
        Chat_tree = np.zeros(N_fft, dtype=complex)
        if tree_groups:
            n_tree_diags = sum(g['n_diagrams'] for g in tree_groups)
            for g in tree_groups:
                Chat_tree += evaluate_kernel_group(g, num_params,
                                                  omega_grid=omega_grid)
            print(f'\n── Tree: {len(tree_groups)} unique kernel(s), '
                  f'{n_tree_diags} diagram(s)  ->  Chat(0) = {Chat_tree[mid]:.6e}')
    else:
        # k>=3: full nD spectrum evaluation + nD IFT
        # For k=3: evaluate S(w0, w1) on a 2D grid, then 2D IFT.
        grid_shape = tuple([N_fft] * n_ext)
        Chat_tree = np.zeros(grid_shape, dtype=complex)
        if tree_groups:
            n_tree_diags = sum(g['n_diagrams'] for g in tree_groups)
            print(f'\n\u2500\u2500 Tree: {len(tree_groups)} unique kernel(s), '
                  f'{n_tree_diags} diagram(s)')
            print(f'   Evaluating on {n_ext}D grid ({N_fft}^{n_ext} = {N_fft**n_ext} points)...')

            for g in tree_groups:
                Chat_tree += evaluate_kernel_group(g, num_params,
                                                  omega_grid=omega_grid)

            mid_idx = tuple([mid] * n_ext)
            print(f'   Chat(0,...,0) = {Chat_tree[mid_idx]:.6e}')

    # ── Step 2: Loop — compute unique loop integrands, ──
    #    multiply by diagram-specific external factors, sum.
    if n_ext <= 1:
        Chat_loop = np.zeros(N_fft, dtype=complex)
    else:
        Chat_loop = np.zeros(grid_shape, dtype=complex)

    if loop_integrand_groups:
        n_unique = len(loop_integrand_groups)
        n_total_diags = sum(
            sum(g['n_diagrams'] for g in gs)
            for gs in loop_integrand_groups.values()
        )
        print(f'\n── Loop: {n_unique} unique loop integrand(s), '
              f'{n_total_diags} diagram(s)')

        # Canonical variable names (matching prop_factors from canonicalize)
        sample_ir = next(iter(loop_integrand_groups.values()))[0]['representative_ir']
        n_ext = len(sample_ir['ext_freqs'])
        ext_vars_canonical = [SR.var(f'w_{ei}') for ei in range(n_ext)]
        ext_var = ext_vars_canonical[0] if ext_vars_canonical else None

        for li, (lsig, groups) in enumerate(loop_integrand_groups.items(), 1):
            ell = groups[0]['loop_number']
            total_diags = sum(g['n_diagrams'] for g in groups)

            # Build and evaluate the loop-only integrand once
            rep = groups[0]
            loop_vars_canonical = [SR.var(f'L_{lv}') for lv in range(ell)]
            loop_factors = [f for f in rep['prop_factors']
                            if _factor_depends_on(f, loop_vars_canonical)]

            loop_product = _build_factor_product(
                loop_factors, propagator_data, omega, num_params)
            loop_var = loop_vars_canonical[0] if loop_vars_canonical else None

            loop_integral = _eval_loop_integral_on_grid(
                loop_product, ext_var, loop_var,
                omega_grid, num_params)

            # For each kernel group sharing this loop integrand,
            # multiply by its external factors and scalar prefactor
            Chat_this_integrand = np.zeros(N_fft, dtype=complex)
            for g in groups:
                ext_factors = [f for f in g['prop_factors']
                               if not _factor_depends_on(f, loop_vars_canonical)]
                ext_product = _build_factor_product(
                    ext_factors, propagator_data, omega, num_params)
                ext_on_grid = _eval_product_on_grid(ext_product, ext_var, omega_grid)

                pf = float(g['combined_prefactor'].subs(num_params))
                Chat_this_integrand += pf * ext_on_grid * loop_integral

            Chat_loop += Chat_this_integrand

            print(f'   Integrand {li}/{n_unique} (ell={ell}, '
                  f'{len(groups)} kernel group(s), '
                  f'{total_diags} diagrams): '
                  f'Chat(0) = {Chat_this_integrand[mid]:.6e}')

    # ── Step 3: Sum + IFT ──
    print(f'\n{"="*60}')
    print('Summary')
    print(f'{"="*60}')

    if n_ext <= 1:
        Chat_total = Chat_tree + Chat_loop
        tau_out, C_tree_tau = inverse_fourier(Chat_tree, omega_grid)
        _, C_total_tau = inverse_fourier(Chat_total, omega_grid)
        C_loop_tau = None
        if not np.all(Chat_loop == 0):
            _, C_loop_tau = inverse_fourier(Chat_loop, omega_grid)
        print(f'  Tree  Chat(0) = {Chat_tree[mid]:.6e}')
        print(f'  Loop  Chat(0) = {Chat_loop[mid]:.6e}')
        print(f'  Total Chat(0) = {Chat_total[mid]:.6e}')
    else:
        # k>=3: full nD IFT, then extract slices
        Chat_total = Chat_tree + Chat_loop
        tau_out, C_tree_tau_full = inverse_fourier(Chat_tree, omega_grid)
        _, C_total_tau_full = inverse_fourier(Chat_total, omega_grid)
        C_loop_tau = None

        # Extract 1D slices (fix other taus at 0 = mid index)
        C_tree_tau_slices = {}
        C_total_tau_slices = {}
        for s in range(n_ext):
            # Build index: all dimensions at mid except dimension s
            idx = [mid] * n_ext
            idx[s] = slice(None)
            idx = tuple(idx)
            C_tree_tau_slices[s] = C_tree_tau_full[idx]
            C_total_tau_slices[s] = C_total_tau_full[idx]

        mid_idx = tuple([mid] * n_ext)
        print(f'  Tree  Chat(0) = {Chat_tree[mid_idx]:.6e}')
        print(f'  Total Chat(0) = {Chat_total[mid_idx]:.6e}')

    # ── Plotting ──
    import matplotlib.pyplot as plt

    if n_ext <= 1:
        # k=2: 1D spectrum + 1D C(tau)
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        ax = axes[0]
        ax.plot(omega_grid, Chat_tree.real, 'b-', lw=1.5, label=r'Tree Re $\hat{C}$')
        if not np.all(Chat_loop == 0):
            ax.plot(omega_grid, Chat_loop.real, 'r-', lw=1.5, label=r'Loop Re $\hat{C}$')
        ax.set_xlabel(r'$\omega$', fontsize=13)
        ax.set_ylabel(r'$\hat{C}^{(' + str(k) + r')}(\omega)$', fontsize=13)
        ax.set_title(f'$k={k}$ Frequency-domain spectrum', fontsize=14)
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(-2, 2)

        ax = axes[1]
        mask = np.abs(tau_out) <= 50
        ax.plot(tau_out[mask], C_tree_tau[mask].real, 'b-', lw=2, label='Tree')
        if C_loop_tau is not None:
            ax.plot(tau_out[mask], C_total_tau[mask].real, 'r--', lw=2, label='Tree + Loop')
        ax.set_xlabel(r'$\tau = t_1 - t_2$', fontsize=13)
        ax.set_ylabel(r'$C^{(' + str(k) + r')}(\tau)$', fontsize=13)
        ax.set_title(f'$k={k}$ Correlator', fontsize=14)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.axhline(0, color='k', lw=0.5)

    else:
        # k>=3: slice plots
        fig, axes = plt.subplots(1, n_ext, figsize=(7 * n_ext, 5))
        if n_ext == 1:
            axes = [axes]

        mask = np.abs(tau_out) <= 50
        for s in range(n_ext):
            ax = axes[s]
            C_tree_s = C_tree_tau_slices[s]
            ax.plot(tau_out[mask], C_tree_s[mask].real, 'b-', lw=2, label='Tree')
            C_total_s = C_total_tau_slices[s]
            if not np.allclose(C_total_s, C_tree_s):
                ax.plot(tau_out[mask], C_total_s[mask].real, 'r--', lw=2, label='Tree + Loop')

            other_labels = ', '.join(
                rf'$\tau_{{{j+1}}}=0$' for j in range(n_ext) if j != s)
            ax.set_xlabel(rf'$\tau_{{{s+1}}}$', fontsize=13)
            ax.set_ylabel(r'$C^{(' + str(k) + r')}$', fontsize=13)
            ax.set_title(f'$k={k}$ slice: vary $\\tau_{{{s+1}}}$ ({other_labels})',
                         fontsize=13)
            ax.legend(fontsize=10)
            ax.grid(True, alpha=0.3)
            ax.axhline(0, color='k', lw=0.5)

    plt.tight_layout()
    plt.show()


### 8. Simulation comparison

Simulate the 2-population nonlinear Hawkes process with the same
fundamental parameters and compare:
- Mean firing rates: simulation vs MF + 1-loop
- Cross-covariance $C_{12}(\tau)$: simulation vs tree + 1-loop

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8.  Simulation of the 2-population nonlinear Hawkes process
# ═══════════════════════════════════════════════════════════════
import numpy as np
from numpy.random import default_rng

# ── Simulation parameters ──
T_sim    = float(200000)    # total simulation time
dt_sim   = float(0.02)     # Euler step (small relative to tau)
dt_bin   = float(0.25)     # bin width for covariance estimation
tau_max  = float(50)       # max lag for covariance
rng      = default_rng(int(42))

# ── Model parameters (from fundamental dict) ──
E_sim   = [float(x) for x in fundamental['E']]
w_sim   = [[float(x) for x in row] for row in fundamental['w']]
tau_sim = float(fundamental['tau'])
a_sim   = [float(x) for x in fundamental['a']]
npop_sim = len(E_sim)


def phi_sim(a_val, v_val):
    """Transfer function: phi(v) = (a/2) v^2."""
    return (a_val / 2.0) * v_val**2


# ═══════════════════════════════════════════════════════════════
# Euler-step simulation with Poisson spike draws
# ═══════════════════════════════════════════════════════════════
# At each timestep dt:
#   1. Compute intensity: lambda_i = phi_i(v_i)
#   2. Draw spikes: n_i ~ Poisson(lambda_i * dt)
#   3. Euler step voltage: dv_i/dt = (-v_i + E_i + sum_j w_ij * n_j/dt) / tau
#      i.e.  v_i += dt/tau * (-v_i + E_i) + sum_j w_ij * n_j / tau

n_steps = int(T_sim / dt_sim)
v = np.array([float(vstar_vals[i]) for i in range(npop_sim)])  # init at MF

# Store binned spike counts for covariance
bin_size_steps = max(int(dt_bin / dt_sim), 1)
n_bins = n_steps // bin_size_steps
binned_counts = np.zeros((npop_sim, n_bins))

total_spikes = np.zeros(npop_sim)
current_bin = 0
steps_in_bin = 0

print(f'Simulating T={T_sim:.0f}, dt={dt_sim}, n_steps={n_steps}...')

for step in range(n_steps):
    # Compute intensity at current voltage
    lam = np.array([phi_sim(a_sim[i], v[i]) for i in range(npop_sim)])
    lam = np.maximum(lam, 0.0)  # ensure non-negative

    # Draw spikes from Poisson(lambda_i * dt)
    spikes = rng.poisson(lam * dt_sim)

    # Accumulate spike counts
    total_spikes += spikes
    if current_bin < n_bins:
        binned_counts[:, current_bin] += spikes

    steps_in_bin += 1
    if steps_in_bin >= bin_size_steps:
        current_bin += 1
        steps_in_bin = 0

    # Euler step voltage:
    #   tau dv/dt = -v + E + sum_j w_ij * spike_train_j(t)
    # where spike_train_j(t) = n_j / dt (instantaneous rate from spikes)
    # So: v += (dt/tau)*(-v + E) + (1/tau) * W @ spikes
    E_arr = np.array(E_sim)
    v += (dt_sim / tau_sim) * (-v + E_arr) + np.dot(w_sim, spikes) / tau_sim

sim_rates = total_spikes / T_sim
print(f'Done.')
for i in range(npop_sim):
    print(f'  Pop {i+1}: {int(total_spikes[i])} spikes, rate = {sim_rates[i]:.4f}')


# ═══════════════════════════════════════════════════════════════
# Compute covariance from binned spike trains
# ═══════════════════════════════════════════════════════════════

# Convert counts to rates
binned_rates = binned_counts / dt_bin

# Subtract mean
for i in range(npop_sim):
    binned_rates[i] -= binned_rates[i].mean()

# ═══════════════════════════════════════════════════════════════
# Compute k-point cumulant from simulation to match theory
# ═══════════════════════════════════════════════════════════════
max_lag_bins = int(tau_max / dt_bin)
n_fft_sim = 2 * n_bins
tau_sim_grid = np.arange(-max_lag_bins, max_lag_bins + 1) * dt_bin
n_ext = k - 1  # number of independent time differences (stationary)

# Population indices for external fields (0-based)
pop_indices = [ef[1] - 1 for ef in external_fields]
field_labels = [f'{ef[0]}_{ef[1]}' for ef in external_fields]
corr_label = ', '.join(field_labels)

def extract_lag_window(xcorr_full, max_lag_bins, n_fft_total):
    out = np.zeros(2 * max_lag_bins + 1)
    for idx_lag, lag in enumerate(range(-max_lag_bins, max_lag_bins + 1)):
        out[idx_lag] = xcorr_full[lag % n_fft_total]
    return out

if k == 1:
    print(f'k=1: mean rates comparison only')
    C_sim_slices = {}

elif k == 2:
    # 2-point: standard cross-correlation
    F_a = np.fft.fft(binned_rates[pop_indices[0]], n=n_fft_sim)
    F_b = np.fft.fft(binned_rates[pop_indices[1]], n=n_fft_sim)
    xcorr = np.fft.ifft(F_a * np.conj(F_b)).real / (n_bins * dt_bin) * dt_bin
    C_sim_slices = {0: extract_lag_window(xcorr, max_lag_bins, n_fft_sim)}

elif k == 3:
    # 3-point cumulant slices.
    # C^(3)(tau1, tau2) = <dn_a(t) dn_b(t+tau1) dn_c(t+tau2)>_c
    # Since means are subtracted, <xyz>_c = <xyz> for zero-mean variables.
    #
    # Slice 0 (vary tau1, tau2=0):
    #   C(tau1, 0) = <dn_a(t) * dn_c(t)  ,  dn_b(t+tau1)>
    #   = cross-corr of [dn_a * dn_c] with dn_b
    # Slice 1 (vary tau2, tau1=0):
    #   C(0, tau2) = <dn_a(t) * dn_b(t)  ,  dn_c(t+tau2)>
    #   = cross-corr of [dn_a * dn_b] with dn_c

    a, b, c = pop_indices[0], pop_indices[1], pop_indices[2]
    C_sim_slices = {}

    for s in range(n_ext):
        if s == 0:
            product = binned_rates[a] * binned_rates[c]  # fields at tau2=0
            F_product = np.fft.fft(product, n=n_fft_sim)
            F_target = np.fft.fft(binned_rates[b], n=n_fft_sim)
        else:
            product = binned_rates[a] * binned_rates[b]  # fields at tau1=0
            F_product = np.fft.fft(product, n=n_fft_sim)
            F_target = np.fft.fft(binned_rates[c], n=n_fft_sim)

        xcorr = np.fft.ifft(F_product * np.conj(F_target)).real / (n_bins * dt_bin) * dt_bin
        C_sim_slices[s] = extract_lag_window(xcorr, max_lag_bins, n_fft_sim)

else:
    print(f'k={k}: simulation cumulant not yet implemented for k>3')
    C_sim_slices = {}

print(f'\nSimulated rates: {[f"{r:.4f}" for r in sim_rates]}')
print(f'MF rates:        {[f"{nstar_vals[i]:.4f}" for i in range(npop_sim)]}')


# ═══════════════════════════════════════════════════════════════
# Comparison plots
# ═══════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

n_plots = max(1, n_ext)
fig, axes = plt.subplots(1, 1 + n_plots, figsize=(7 * (1 + n_plots), 5))
if not isinstance(axes, np.ndarray):
    axes = [axes]

# ── Panel 0: Mean rates ──
ax = axes[0]
x_pos = np.arange(npop_sim)
width = 0.25
ax.bar(x_pos - width, sim_rates, width, label='Simulation',
       color='#4CAF50', edgecolor='k', linewidth=0.5)
mf_rates = [float(nstar_vals[i]) for i in range(npop_sim)]
ax.bar(x_pos, mf_rates, width, label='Mean-field',
       color='#2196F3', edgecolor='k', linewidth=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels([f'Pop {i+1}' for i in range(npop_sim)])
ax.set_ylabel('Firing rate', fontsize=12)
ax.set_title('Mean firing rates', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# ── Cumulant slice panels ──
for s in range(n_ext):
    ax = axes[1 + s]

    # Simulation
    if s in C_sim_slices:
        ax.plot(tau_sim_grid, C_sim_slices[s], 'k-', lw=1.5, alpha=0.7,
                label='Simulation')

    # Theory
    mask_an = np.abs(tau_out) <= tau_max
    if n_ext <= 1:
        ax.plot(tau_out[mask_an], C_tree_tau[mask_an].real, 'b-', lw=2,
                label='Tree')
        if C_loop_tau is not None:
            ax.plot(tau_out[mask_an], C_total_tau[mask_an].real, 'r--', lw=2,
                    label='Tree + 1-loop')
    else:
        if s in C_tree_tau_slices:
            ax.plot(tau_out[mask_an], C_tree_tau_slices[s][mask_an].real,
                    'b-', lw=2, label='Tree')
        if s in C_total_tau_slices:
            C_tot_s = C_total_tau_slices[s][mask_an].real
            C_tree_s = C_tree_tau_slices[s][mask_an].real
            if not np.allclose(C_tot_s, C_tree_s):
                ax.plot(tau_out[mask_an], C_tot_s, 'r--', lw=2,
                        label='Tree + 1-loop')

    # Labels
    if n_ext == 1:
        ax.set_xlabel(r'$\tau = t_1 - t_2$', fontsize=13)
        is_auto = (external_fields[0] == external_fields[1])
        ctype = 'Auto' if is_auto else 'Cross'
        ax.set_title(f'{ctype}-covariance: {corr_label}', fontsize=13)
    else:
        other = ', '.join(rf'$\tau_{{{j+1}}}=0$' for j in range(n_ext) if j != s)
        ax.set_xlabel(rf'$\tau_{{{s+1}}}$', fontsize=13)
        ax.set_title(f'$k={k}$ cumulant slice: {other}', fontsize=13)

    ax.set_ylabel(rf'$C^{{({k})}}$', fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', lw=0.5)

plt.tight_layout()
plt.show()

# ── Summary ──
print(f'\n{"="*60}')
print('Comparison summary')
print(f'{"="*60}')
print(f'  k = {k}, external_fields = {external_fields}')
for i in range(npop_sim):
    mf = nstar_vals[i]
    sim_r = sim_rates[i]
    pct = 100 * (sim_r - mf) / mf if mf > 0 else 0
    print(f'  Pop {i+1}:  sim={sim_r:.4f}  MF={mf:.4f}  diff={pct:+.2f}%')
print(f'\n  T={T_sim:.0f}, dt={dt_sim}, bins={n_bins}')
